In [1]:
!rm .fleche -rf

# Getting Started with Fleche

This notebook demonstrates the main features of the `fleche` library, a caching library for Python.

## Long-running calculation

In [2]:
import time
from fleche import fleche, cache, tags, project
from fleche.digest import Digest

In [3]:
@fleche
def long_running_calculation(x):
    print(f'Running calculation for {x}...')
    time.sleep(2)
    return x * x

In [4]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'First call took {end - start:.2f} seconds.')

Running calculation for 2...
First call took 2.00 seconds.


In [5]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'Second call took {end - start:.2f} seconds.')

Second call took 0.00 seconds.


As you can see, the second call returns almost instantly, because the result was cached.

## Recursive function

In [6]:
@fleche
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

In [7]:
start = time.time()
fib(20)
end = time.time()
print(f'fib(20) took {end - start:.4f} seconds with caching.')

fib(20) took 0.0517 seconds with caching.


Without caching, this would be much slower as each call to `fib` would be recomputed.

## Caching Methods of User-defined Types

`fleche` can also cache methods of classes. For this to work, the class must be "digest-compatible". You can make a class digest-compatible by implementing a `__digest__` method or by using a `dataclass`.

In [ ]:
class MyClass:
    def __init__(self, val):
        self.val = val
    
    def __digest__(self):
        # The digest defines how the instance is identified in the cache
        return Digest(str(self.val))

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

In [ ]:
obj = MyClass(10)

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

If you mutate the instance such that its digest changes, the cache will be missed.

In [ ]:
obj.val = 20
start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Call after mutation took {time.time() - start:.2f} seconds.")

## Metadata

`fleche` allows you to add metadata to your cached functions using the `tags` context manager. This can be useful for organizing and querying your results.

In [8]:
@fleche
def another_calculation(a, b):
    return a + b

In [9]:
with tags(project='my_project', category='testing'):
    another_calculation(1, 2)
    another_calculation(3, 4)

This metadata is stored alongside the cached result. You can then use the `metadata.table` method to view the metadata for all cached results.

In [10]:
cache().metadata.table()

,timestart,timestop,walltime,arguments,result,name,module,version,project,category
70b0891542b04fd7a9cbd0c5204a63c35a6f397d341ee61d541dc581d6cb8500,1.770248e+09,1.770248e+09,2.000397,{0: 'dbc1b4c900ffe48d575b5da5c638040125f65db0f...,e52d9c508c502347344d8c07ad91cbd6068afc75ff6292...,long_running_calculation,__main__,None,NaN,NaN
22fb5c2780e8254d8b32c864c73f4395fdffbe2f986fd1e0c0bea4a734a02916,1.770248e+09,1.770248e+09,0.000013,{0: '4bf5122f344554c53bde2ebb8cd2b7e3d1600ad63...,4bf5122f344554c53bde2ebb8cd2b7e3d1600ad631c385...,fib,__main__,None,NaN,NaN
08316c89bb9bde1721a11225215f50e5585502f7f7e97d3783ccb92e51da9af9,1.770248e+09,1.770248e+09,0.000026,{0: '6e340b9cffb37a989ca544e6bb780a2c78901d3fb...,6e340b9cffb37a989ca544e6bb780a2c78901d3fb33738...,fib,__main__,None,NaN,NaN
5ed50f0f5e3d7fd468931c7de68c4addeb1e419cd8868d7bb38f84357373348a,1.770248e+09,1.770248e+09,0.005699,{0: 'dbc1b4c900ffe48d575b5da5c638040125f65db0f...,4bf5122f344554c53bde2ebb8cd2b7e3d1600ad631c385...,fib,__main__,None,NaN,NaN
5fb55efa651989da640b4cbc4763f1333e37463b3f8e6d335f705650e2ea19cf,1.770248e+09,1.770248e+09,0.008323,{0: '084fed08b978af4d7d196a7446a86b58009e636b6...,dbc1b4c900ffe48d575b5da5c638040125f65db0fe3e24...,fib,__main__,None,NaN,NaN
84cfeff0d9a08e011f1273729a76adae6db7e4f08035fc701f871da2cb29f73e,1.770248e+09,1.770248e+09,0.011005,{0: 'e52d9c508c502347344d8c07ad91cbd6068afc75f...,084fed08b978af4d7d196a7446a86b58009e636b611db1...,fib,__main__,None,NaN,NaN
a34e82b8288a2222e6295635a818b4751f8bc93728c4d6f91a56088adb9ea40a,1.770248e+09,1.770248e+09,0.013701,{0: 'e77b9a9ae9e30b0dbdb6f510a264ef9de781501d7...,e77b9a9ae9e30b0dbdb6f510a264ef9de781501d7b6b92...,fib,__main__,None,NaN,NaN
d46bbde636a78849e93c8fb14f9b259aae038ac0cea58f63e8659352e95761f8,1.770248e+09,1.770248e+09,0.016167,{0: '67586e98fad27da0b9968bc039a1ef34c939b9b8e...,beead77994cf573341ec17b58bbf7eb34d2711c993c1d9...,fib,__main__,None,NaN,NaN
696feb5c63724cc43b06fca2ab4a0fbabe669c69aa477fa09f101477314bf634,1.770248e+09,1.770248e+09,0.018594,{0: 'ca358758f6d27e6cf45272937977a748fd88391db...,9d1e0e2d9459d06523ad13e28a4093c2316baafe7aec5b...,fib,__main__,None,NaN,NaN
f84b1b6ce87802f9f8f8ac1b9e385b97b9baa1bc4b13f6ac5061860d55624428,1.770248e+09,1.770248e+09,0.021226,{0: 'beead77994cf573341ec17b58bbf7eb34d2711c99...,2f0fd1e89b8de1d57292742ec380ea47066e307ad645f5...,fib,__main__,None,NaN,NaN


## Filtering

The metadata table is just pandas so you can query and filter as you like.

In [13]:
cache().metadata.table().query('name!="fib"')

,timestart,timestop,walltime,arguments,result,name,module,version,project,category
70b0891542b04fd7a9cbd0c5204a63c35a6f397d341ee61d541dc581d6cb8500,1.770248e+09,1.770248e+09,2.000397,{0: 'dbc1b4c900ffe48d575b5da5c638040125f65db0f...,e52d9c508c502347344d8c07ad91cbd6068afc75ff6292...,long_running_calculation,__main__,None,NaN,NaN
eff3d8bc69df1bfbdcc161bc3fb0f4675766ab121254d681185eb13fa3020175,1.770248e+09,1.770248e+09,0.000041,{0: '4bf5122f344554c53bde2ebb8cd2b7e3d1600ad63...,084fed08b978af4d7d196a7446a86b58009e636b611db1...,another_calculation,__main__,None,my_project,testing
4848e14a28b588633a55dcbc2b8700beff01745aa785ea0c76a6f512801997fb,1.770248e+09,1.770248e+09,0.000035,{0: '084fed08b978af4d7d196a7446a86b58009e636b6...,ca358758f6d27e6cf45272937977a748fd88391db679ce...,another_calculation,__main__,None,my_project,testing
51b5cb6fa3e8e693ac45e7ddf1dcaa868c79f78e5a30ae9b27068187293e280e,1.770248e+09,1.770248e+09,0.000065,{0: 'e77b9a9ae9e30b0dbdb6f510a264ef9de781501d7...,e7cf46a078fed4fafd0b5e3aff144802b853f8ae459a4f...,another_calculation,__main__,None,another_project,NaN


In [14]:
cache().metadata.table().query('project=="my_project"')

,timestart,timestop,walltime,arguments,result,name,module,version,project,category
eff3d8bc69df1bfbdcc161bc3fb0f4675766ab121254d681185eb13fa3020175,1.770248e+09,1.770248e+09,0.000041,{0: '4bf5122f344554c53bde2ebb8cd2b7e3d1600ad63...,084fed08b978af4d7d196a7446a86b58009e636b611db1...,another_calculation,__main__,None,my_project,testing
4848e14a28b588633a55dcbc2b8700beff01745aa785ea0c76a6f512801997fb,1.770248e+09,1.770248e+09,0.000035,{0: '084fed08b978af4d7d196a7446a86b58009e636b6...,ca358758f6d27e6cf45272937977a748fd88391db679ce...,another_calculation,__main__,None,my_project,testing
